In [1]:
# =============================================================================
# **NOTEBOOK 1: DATA INGESTION & STORAGE DESIGN**
# **US Accidents Severity Classification - Big Data Assignment**
# **Dataset:** US_Accidents_March23.csv (~1.6GB, 4.2M rows)
# **Target:** Use full dataset for analysis
# =============================================================================

# Install required packages if not already installed
import subprocess, sys

packages = [
    "pyspark==3.5.0",
    "pyarrow",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "pyyaml",
    "findspark"
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages installed successfully.")


All packages installed successfully.


In [2]:
# =============================================================================
# **SECTION 1: SPARKSESSION CONFIGURATION**
# Configured for local mode - adjust master/executor settings for cluster
# =============================================================================

import os
import yaml
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, FloatType,
    IntegerType, DoubleType, BooleanType, TimestampType
)
from pyspark import StorageLevel
import logging

# Configure Python logging for data lineage tracking
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler()]
)
logger = logging.getLogger("DataIngestion")
# Mask home directory in all log output so no username appears
class _HomeFilter(logging.Filter):
    _h = os.path.expanduser("~")
    def filter(self, r):
        try:
            r.msg = r.getMessage().replace(self._h, "~")
            r.args = None
        except Exception:
            pass
        return True
logging.getLogger().addFilter(_HomeFilter())


# Load Spark configuration from YAML
CONFIG_PATH = os.path.join(os.path.dirname(os.getcwd()), "config", "spark_config.yaml")
if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r") as f:
        spark_cfg = yaml.safe_load(f)
    logger.info("Loaded spark_config.yaml successfully.")
else:
    # Fallback defaults if config file not found
    spark_cfg = {
        "app_name": "USAccidentsSeverityClassification",
        "master": "local[*]",
        "executor_memory": "4g",
        "driver_memory": "4g",
        "executor_cores": "4",
        "shuffle_partitions": "200",
        "broadcast_threshold": "20971520",
        "kryo_serializer": "true"
    }
    logger.warning("spark_config.yaml not found, using fallback defaults.")

# Build SparkSession with performance-optimized settings
spark = (
    SparkSession.builder
    .appName(spark_cfg["app_name"])
    .master(spark_cfg["master"])
    .config("spark.executor.memory", spark_cfg["executor_memory"])
    .config("spark.driver.memory", spark_cfg["driver_memory"])
    .config("spark.executor.cores", spark_cfg["executor_cores"])
    .config("spark.sql.shuffle.partitions", spark_cfg["shuffle_partitions"])
    .config("spark.sql.autoBroadcastJoinThreshold", spark_cfg["broadcast_threshold"])
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.kryoserializer.buffer.max", "512m")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
    .config("spark.sql.parquet.enableVectorizedReader", "true")
    .config("spark.sql.parquet.mergeSchema", "false")
    .config("spark.ui.enabled", "true")
    .config("spark.ui.port", "4040")
    .config("spark.checkpoint.dir", "/tmp/spark_checkpoints")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate()
)

# Set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

# Print Spark version and resource summary
print(f"Spark Version      : {spark.version}")
print(f"Application Name   : {spark.sparkContext.appName}")
print(f"Master             : {spark.sparkContext.master}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
logger.info("SparkSession initialized successfully.")


2026-03-03 03:35:40,283 [WARNING] spark_config.yaml not found, using fallback defaults.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 03:35:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-03-03 03:35:43,624 [INFO] SparkSession initialized successfully.


Spark Version      : 3.5.0
Application Name   : USAccidentsSeverityClassification
Master             : local[*]
Default Parallelism: 4


In [3]:
# =============================================================================
# **SECTION 2: DEFINE SCHEMA FOR DATA VALIDATION AT INGESTION**
# Explicit schema avoids expensive schema inference on 4.2M row dataset
# =============================================================================

# Define complete schema for US Accidents dataset (46 columns)
accidents_schema = StructType([
    StructField("ID", StringType(), True),
    StructField("Source", StringType(), True),
    StructField("Severity", IntegerType(), True),
    StructField("Start_Time", TimestampType(), True),
    StructField("End_Time", TimestampType(), True),
    StructField("Start_Lat", DoubleType(), True),
    StructField("Start_Lng", DoubleType(), True),
    StructField("End_Lat", DoubleType(), True),
    StructField("End_Lng", DoubleType(), True),
    StructField("Distance(mi)", DoubleType(), True),
    StructField("Description", StringType(), True),
    StructField("Street", StringType(), True),
    StructField("City", StringType(), True),
    StructField("County", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Zipcode", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("Timezone", StringType(), True),
    StructField("Airport_Code", StringType(), True),
    StructField("Weather_Timestamp", TimestampType(), True),
    StructField("Temperature(F)", DoubleType(), True),
    StructField("Wind_Chill(F)", DoubleType(), True),
    StructField("Humidity(%)", DoubleType(), True),
    StructField("Pressure(in)", DoubleType(), True),
    StructField("Visibility(mi)", DoubleType(), True),
    StructField("Wind_Direction", StringType(), True),
    StructField("Wind_Speed(mph)", DoubleType(), True),
    StructField("Precipitation(in)", DoubleType(), True),
    StructField("Weather_Condition", StringType(), True),
    StructField("Amenity", BooleanType(), True),
    StructField("Bump", BooleanType(), True),
    StructField("Crossing", BooleanType(), True),
    StructField("Give_Way", BooleanType(), True),
    StructField("Junction", BooleanType(), True),
    StructField("No_Exit", BooleanType(), True),
    StructField("Railway", BooleanType(), True),
    StructField("Roundabout", BooleanType(), True),
    StructField("Station", BooleanType(), True),
    StructField("Stop", BooleanType(), True),
    StructField("Traffic_Calming", BooleanType(), True),
    StructField("Traffic_Signal", BooleanType(), True),
    StructField("Turning_Loop", BooleanType(), True),
    StructField("Sunrise_Sunset", StringType(), True),
    StructField("Civil_Twilight", StringType(), True),
    StructField("Nautical_Twilight", StringType(), True),
    StructField("Astronomical_Twilight", StringType(), True),
])

logger.info(f"Schema defined with {len(accidents_schema.fields)} fields.")
print(f"Schema field count: {len(accidents_schema.fields)}")


2026-03-03 03:35:43,636 [INFO] Schema defined with 46 fields.


Schema field count: 46


In [4]:
# =============================================================================
# **SECTION 3: DATA INGESTION - LOAD RAW CSV**
# Full dataset: 1.6GB, 4.2M rows — entire dataset used for analysis
# =============================================================================

import time

# Define paths
import pathlib as _pl
_cwd = _pl.Path(os.getcwd()).resolve()
BASE_DIR = (str(_cwd) if (_cwd / "US_Accidents_March23.csv").exists()
            else str(_cwd.parent.parent) if (_cwd.parent.parent / "US_Accidents_March23.csv").exists()
            else str(_cwd.parent.parent))
RAW_CSV_PATH = os.path.join(BASE_DIR, "US_Accidents_March23.csv")
PARQUET_FULL_PATH = os.path.join(BASE_DIR, "code", "data", "samples", "accidents_full.parquet")
PARQUET_SAMPLED_PATH = os.path.join(BASE_DIR, "code", "data", "samples", "accidents_sampled.parquet")
SCHEMA_PATH = os.path.join(BASE_DIR, "code", "data", "schemas", "accidents_schema.json")

logger.info("Starting data ingestion...")

# Ingestion with error handling
try:
    ingestion_start = time.time()

    # Load full CSV with explicit schema
    raw_df = (
        spark.read
        .option("header", "true")
        .option("mode", "PERMISSIVE")          # Keep malformed rows, mark with null
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .schema(accidents_schema)
        .csv(RAW_CSV_PATH)
    )

    ingestion_end = time.time()
    logger.info(f"Raw CSV loaded in {ingestion_end - ingestion_start:.2f}s")

except Exception as e:
    logger.error(f"Ingestion failed: {e}")
    raise

# Data Lineage: document ingestion step
lineage = {
    "step": "ingestion",
    "source": RAW_CSV_PATH,
    "format": "CSV",
    "schema_enforced": True,
    "mode": "PERMISSIVE",
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S")
}

# Count total rows (triggers first Spark action)
total_rows = raw_df.count()
total_cols = len(raw_df.columns)
print(f"Total rows loaded : {total_rows:,}")
print(f"Total columns     : {total_cols}")
print(f"Ingestion time    : {ingestion_end - ingestion_start:.2f}s")


2026-03-03 03:35:43,656 [INFO] Starting data ingestion...
2026-03-03 03:35:45,104 [INFO] Raw CSV loaded in 1.45s


Total rows loaded : 4,209,699
Total columns     : 46
Ingestion time    : 1.45s


In [5]:
# =============================================================================
# **SECTION 4: DATA VALIDATION AT INGESTION**
# Validate schema conformance, null counts, and row integrity
# =============================================================================

# Persist raw_df at MEMORY_AND_DISK level for reuse in validation
raw_df.persist(StorageLevel.MEMORY_AND_DISK)
logger.info("raw_df persisted at MEMORY_AND_DISK level.")

# Validation 1: Check for null counts in critical columns
critical_cols = ["Severity", "Start_Time", "Start_Lat", "Start_Lng", "State"]

null_counts = raw_df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in critical_cols]
).collect()[0].asDict()

print("\n=== NULL COUNT VALIDATION (Critical Columns) ===")
for col_name, null_count in null_counts.items():
    pct = (null_count / total_rows) * 100
    status = "PASS" if pct < 5 else "WARN"
    print(f"  [{status}] {col_name}: {null_count:,} nulls ({pct:.2f}%)")

# Validation 2: Severity must be in range [1, 4]
invalid_severity = raw_df.filter(
    ~F.col("Severity").isin([1, 2, 3, 4])
).count()
print(f"\n  [{'PASS' if invalid_severity == 0 else 'FAIL'}] Invalid Severity values: {invalid_severity:,}")

# Validation 3: Coordinate range check (continental US bounding box)
invalid_coords = raw_df.filter(
    (F.col("Start_Lat") < 24.0) | (F.col("Start_Lat") > 72.0) |
    (F.col("Start_Lng") < -170.0) | (F.col("Start_Lng") > -66.0)
).count()
print(f"  [{'PASS' if invalid_coords < total_rows * 0.01 else 'WARN'}] Out-of-range coordinates: {invalid_coords:,}")

# Validation 4: Check class distribution
print("\n=== CLASS DISTRIBUTION (Severity) ===")
raw_df.groupBy("Severity").count().orderBy("Severity").show()

logger.info("Data validation completed successfully.")


26/03/03 03:35:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
2026-03-03 03:35:48,271 [INFO] raw_df persisted at MEMORY_AND_DISK level.



=== NULL COUNT VALIDATION (Critical Columns) ===
  [PASS] Severity: 0 nulls (0.00%)
  [PASS] Start_Time: 0 nulls (0.00%)
  [PASS] Start_Lat: 0 nulls (0.00%)
  [PASS] Start_Lng: 0 nulls (0.00%)
  [PASS] State: 0 nulls (0.00%)



  [PASS] Invalid Severity values: 0
  [PASS] Out-of-range coordinates: 0

=== CLASS DISTRIBUTION (Severity) ===


2026-03-03 03:36:29,942 [INFO] Data validation completed successfully.          


+--------+-------+
|Severity|  count|
+--------+-------+
|       1| 376989|
|       2|2583530|
|       3| 823710|
|       4| 425470|
+--------+-------+



In [6]:
# =============================================================================
# **SECTION 5: REPARTITION & PERSIST FULL DATASET**
# No sampling applied — full 1.6 GB / 4.2M row dataset is used for analysis
# Repartition by State aligns partitions with geographic query patterns
# =============================================================================

RANDOM_SEED = 42

logger.info("Using full dataset — no sampling applied.")

# Use full dataset; repartition for downstream query alignment
sampled_df = raw_df.repartition(10, "State")
logger.info("Repartitioned full dataset into 10 partitions by State.")

# Persist for downstream reuse
sampled_df.persist(StorageLevel.MEMORY_AND_DISK)

sampled_count = sampled_df.count()
print(f"Total row count   : {sampled_count:,}")
print(f"Sampling          : None (full dataset)")
print(f"Dataset size      : ~1.6 GB")
print(f"Partition count   : {sampled_df.rdd.getNumPartitions()}")

# Unpersist raw_df to free memory (sampled_df holds the repartitioned reference)
raw_df.unpersist()
logger.info("raw_df unpersisted after repartitioning.")

# Verify class distribution
print("\n=== CLASS DISTRIBUTION (Severity) — Full Dataset ===")
sampled_df.groupBy("Severity").count().orderBy("Severity").show()


2026-03-03 03:36:29,952 [INFO] Using full dataset — no sampling applied.
2026-03-03 03:36:29,965 [INFO] Repartitioned full dataset into 10 partitions by State.
2026-03-03 03:36:52,727 [INFO] raw_df unpersisted after repartitioning.         


Total row count   : 4,209,699
Sampling          : None (full dataset)
Dataset size      : ~1.6 GB
Partition count   : 10

=== CLASS DISTRIBUTION (Severity) — Full Dataset ===
+--------+-------+
|Severity|  count|
+--------+-------+
|       1| 376989|
|       2|2583530|
|       3| 823710|
|       4| 425470|
+--------+-------+



In [7]:
# =============================================================================
# **SECTION 6: BROADCAST JOIN - STATE LOOKUP TABLE ENRICHMENT**
# Small lookup table (<= 20MB) is broadcast to all executors
# Avoids expensive shuffle join; optimal for dimension table enrichment
# =============================================================================

import pandas as pd

# Create state metadata lookup (small table - ideal for broadcast join)
# Maps US state abbreviations to region and timezone categories
state_lookup_data = {
    "CA": ("West", "Pacific"), "TX": ("South", "Central"),
    "FL": ("South", "Eastern"), "NY": ("Northeast", "Eastern"),
    "PA": ("Northeast", "Eastern"), "OH": ("Midwest", "Eastern"),
    "IL": ("Midwest", "Central"), "GA": ("South", "Eastern"),
    "NC": ("South", "Eastern"), "MI": ("Midwest", "Eastern"),
    "VA": ("South", "Eastern"), "WA": ("West", "Pacific"),
    "CO": ("West", "Mountain"), "AZ": ("West", "Mountain"),
    "NV": ("West", "Pacific"), "OR": ("West", "Pacific"),
    "MN": ("Midwest", "Central"), "TN": ("South", "Central"),
    "MO": ("Midwest", "Central"), "MA": ("Northeast", "Eastern"),
    "IN": ("Midwest", "Eastern"), "MD": ("Northeast", "Eastern"),
    "WI": ("Midwest", "Central"), "NJ": ("Northeast", "Eastern"),
    "SC": ("South", "Eastern"), "KY": ("South", "Eastern"),
    "AL": ("South", "Central"), "LA": ("South", "Central"),
    "KS": ("Midwest", "Central"), "OK": ("South", "Central"),
    "UT": ("West", "Mountain"), "NM": ("West", "Mountain"),
    "ID": ("West", "Mountain"), "MT": ("West", "Mountain"),
    "WY": ("West", "Mountain"), "ND": ("Midwest", "Central"),
    "SD": ("Midwest", "Central"), "NE": ("Midwest", "Central"),
    "IA": ("Midwest", "Central"), "AR": ("South", "Central"),
    "MS": ("South", "Central"), "DC": ("Northeast", "Eastern"),
    "CT": ("Northeast", "Eastern"), "RI": ("Northeast", "Eastern"),
    "NH": ("Northeast", "Eastern"), "VT": ("Northeast", "Eastern"),
    "ME": ("Northeast", "Eastern"), "DE": ("Northeast", "Eastern"),
    "WV": ("South", "Eastern"), "AK": ("West", "Alaska"),
    "HI": ("West", "Hawaii"), "PR": ("South", "Atlantic")
}

state_rows = [(k, v[0], v[1]) for k, v in state_lookup_data.items()]
state_lookup_pd = pd.DataFrame(state_rows, columns=["State", "Region", "Climate_Zone"])

# Convert to Spark DataFrame
state_lookup_spark = spark.createDataFrame(state_lookup_pd)

# Broadcast the small lookup table (~4KB) to all worker nodes
from pyspark.sql.functions import broadcast

logger.info("Applying broadcast join with state lookup table.")
enriched_df = sampled_df.join(
    broadcast(state_lookup_spark),
    on="State",
    how="left"
)

# Confirm enrichment columns added
print("Columns after broadcast join enrichment:")
print([col for col in enriched_df.columns if col in ["State", "Region", "Climate_Zone"]])
print(f"Total columns after enrichment: {len(enriched_df.columns)}")
logger.info("Broadcast join completed. Region and Climate_Zone added.")


2026-03-03 03:36:54,039 [INFO] Applying broadcast join with state lookup table.
2026-03-03 03:36:54,072 [INFO] Broadcast join completed. Region and Climate_Zone added.


Columns after broadcast join enrichment:
['State', 'Region', 'Climate_Zone']
Total columns after enrichment: 48


In [8]:
# =============================================================================
# **SECTION 7: WRITE TO PARQUET - STORAGE FORMAT JUSTIFICATION**
# Parquet chosen over CSV/JSON for:
#   - Columnar format: reads only needed columns (80% I/O reduction)
#   - Built-in compression (Snappy): ~3x size reduction vs CSV
#   - Predicate pushdown: avoids reading irrelevant row groups
#   - Native PySpark/MLlib support without re-serialization
# Partitioned by "State" to align with geographic query patterns
# =============================================================================

import json

logger.info("Writing enriched sampled DataFrame to Parquet...")

try:
    write_start = time.time()

    # Write sampled enriched data to Parquet
    (
        enriched_df.write
        .mode("overwrite")
        .partitionBy("State")
        .option("compression", "snappy")
        .parquet(PARQUET_SAMPLED_PATH)
    )

    write_end = time.time()

except Exception as e:
    logger.error(f"Parquet write failed: {e}")
    raise

# Save schema to JSON for downstream notebooks
schema_dict = enriched_df.schema.jsonValue()
os.makedirs(os.path.dirname(SCHEMA_PATH), exist_ok=True)
with open(SCHEMA_PATH, "w") as f:
    json.dump(schema_dict, f, indent=2)

# Verify Parquet write by reading back
verification_df = spark.read.parquet(PARQUET_SAMPLED_PATH)
print(f"\nParquet verification:")
print(f"  Rows read back    : {verification_df.count():,}")
print(f"  Schema consistent : {verification_df.schema == enriched_df.schema}")
print(f"  Write time        : {write_end - write_start:.2f}s")

# List partition directories
import glob
partition_dirs = [d for d in glob.glob(f"{PARQUET_SAMPLED_PATH}/State=*") if os.path.isdir(d)]
print(f"  Partition count   : {len(partition_dirs)} State partitions")

# Unpersist sampled_df and enriched_df after write
sampled_df.unpersist()
enriched_df.unpersist()
logger.info("sampled_df and enriched_df unpersisted after Parquet write.")


2026-03-03 03:36:54,082 [INFO] Writing enriched sampled DataFrame to Parquet...



Parquet verification:


2026-03-03 03:37:22,995 [INFO] sampled_df and enriched_df unpersisted after Parquet write.


  Rows read back    : 4,209,699
  Schema consistent : False
  Write time        : 28.29s
  Partition count   : 49 State partitions


In [9]:
# =============================================================================
# **SECTION 8: PERFORMANCE PROFILING & DATA LINEAGE SUMMARY**
# Capture Spark stage metrics for Spark UI evidence
# =============================================================================

import json

# Collect Spark metrics via sparkContext
sc = spark.sparkContext

print("=== PIPELINE PERFORMANCE SUMMARY ===")
print(f"  Spark App ID      : {sc.applicationId}")
print(f"  Default partition : {sc.defaultParallelism}")
print(f"  Shuffle partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")
print(f"  Broadcast thresh  : {int(spark.conf.get('spark.sql.autoBroadcastJoinThreshold'))//1024//1024} MB")
print(f"  Adaptive enabled  : {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"  Time parser policy: {spark.conf.get('spark.sql.legacy.timeParserPolicy')}")

# Data Lineage Summary
pipeline_lineage = [
    {"step": 1, "name": "CSV Ingestion",    "source": RAW_CSV_PATH,        "rows": total_rows,    "format": "CSV"},
    {"step": 2, "name": "Validation",       "source": "schema+constraints", "rows": total_rows,    "format": "DataFrame"},
    {"step": 3, "name": "Repartition",      "source": "full dataset",       "rows": sampled_count, "format": "DataFrame"},
    {"step": 4, "name": "Broadcast Enrich", "source": "state_lookup",       "rows": sampled_count, "format": "DataFrame"},
    {"step": 5, "name": "Parquet Write",    "source": PARQUET_SAMPLED_PATH, "rows": sampled_count, "format": "Parquet/Snappy"},
]

print("\n=== DATA LINEAGE PIPELINE ===")
for step in pipeline_lineage:
    print(f"  Step {step['step']}: {step['name']:<20} | Rows: {step['rows']:>9,} | Format: {step['format']}")

# Save lineage to JSON for audit
lineage_path = os.path.join(BASE_DIR, "code", "data", "samples", "pipeline_lineage.json")
with open(lineage_path, "w") as f:
    json.dump(pipeline_lineage, f, indent=2)

print("Next step: Run notebook 2_feature_engineering.ipynb")


=== PIPELINE PERFORMANCE SUMMARY ===
  Spark App ID      : local-1772508943211
  Default partition : 4
  Shuffle partitions: 200
  Broadcast thresh  : 20 MB
  Adaptive enabled  : true
  Time parser policy: LEGACY

=== DATA LINEAGE PIPELINE ===
  Step 1: CSV Ingestion        | Rows: 4,209,699 | Format: CSV
  Step 2: Validation           | Rows: 4,209,699 | Format: DataFrame
  Step 3: Repartition          | Rows: 4,209,699 | Format: DataFrame
  Step 4: Broadcast Enrich     | Rows: 4,209,699 | Format: DataFrame
  Step 5: Parquet Write        | Rows: 4,209,699 | Format: Parquet/Snappy
Next step: Run notebook 2_feature_engineering.ipynb
